# Lab 5: Text Embeddings

**Business question:** Can related customer texts share meaning without sharing words?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | Example sentences and the 24 customer messages from Lab 1 | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | Women's clothing reviews | about 40–55 min | Four exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

You do not download or run a language model in this lab. I ran a pretrained embedding model once and saved its vectors in `data/embeddings`. You load those saved vectors, so everyone gets the same numbers. Saved results like these are often called a **cache**.

## Part 1 Guided core

### 1.1 Load the libraries

Select the `ban6403` kernel, then run the cell below. It loads the libraries we use in this lab.

In [ ]:
from pathlib import Path   # file and folder paths
import json                # reads the manifest files
import numpy as np         # arrays of numbers: the vectors
import pandas as pd        # tables
from sklearn.feature_extraction.text import TfidfVectorizer  # the Module 4 word features

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the data is one folder up, in `../data`. The saved vectors are in `../data/embeddings`. Your own results go to `../outputs/m05`.

In [ ]:
# Folder paths, relative to the notebooks folder.
DATA = Path("../data")
EMB = DATA / "embeddings"
OUTPUT_DIR = Path("../outputs/m05")
# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 Read the manifest first

Each set of saved vectors comes with a **manifest**: a small file that says which model made the vectors and how. Read it before you use the numbers. Expect the model `sentence-transformers/all-MiniLM-L6-v2`, 384 dimensions and a limit of 256 tokens.

In [ ]:
# The manifest is a JSON file; json.loads turns its text into a Python dictionary.
manifest = json.loads((EMB / "m05_sentences_minilm_manifest.json").read_text())

# Print the fields that describe how the vectors were made, one per line.
for key in ["model", "model_revision", "dimensions", "max_tokens", "pooling", "normalized", "input"]:
    print(f"{key}: {manifest[key]}")

### 1.4 Load the example sentences and their vectors

The file `m05_demo_sentences.csv` holds 11 short sentences we wrote. The saved vectors are in the same order as the rows of the file: the first vector belongs to S01, the second to S02, and so on. The vectors are stored as compact 16-bit numbers, so `.astype("float32")` turns them into ordinary numbers for computing.

In [ ]:
sentences = pd.read_csv(DATA / "raw/m05_demo_sentences.csv")

# np.load reads a saved numpy array: one row of numbers per sentence.
sent_vectors = np.load(EMB / "m05_sentences_minilm.npy").astype("float32")

# .shape gives (number of rows, number of columns).
print("shape:", sent_vectors.shape)

### 1.5 Look at one vector

A sentence vector is a list of 384 numbers. The columns have no names, so you cannot read one number the way you read a TF-IDF weight. Each vector also has length 1, which makes the next step simple.

In [ ]:
# [0, :8] means row 0 (sentence S01), first 8 columns.
print("first 8 numbers of S01:", sent_vectors[0, :8].round(3))

# np.linalg.norm with axis=1 gives the length of each row; [:4] shows the first four.
print("vector lengths:", np.linalg.norm(sent_vectors, axis=1).round(3)[:4])

### 1.6 Cosine similarity between sentences

For vectors of length 1, **cosine similarity** is just the dot product. It is close to 1 when two vectors point the same way. We compare each anchor sentence (S01, S06, S09) with the sentences written to go with it.

**Similarity in code.** A few pieces of numpy code do most of the work in this lab and in the next modules:

- `A.T` swaps rows and columns (the transpose). `sent_vectors` has shape (11, 384), so `sent_vectors.T` has shape (384, 11).
- `A @ B` is a matrix product. An (n × 384) table `@` a (384 × m) table gives an n × m table. With length-1 vectors, the value in row i, column j is the cosine between item i of A and item j of B. So `sent_vectors @ sent_vectors.T` is an 11 × 11 table of cosines.
- `np.argsort(-scores)` lists positions from the highest score to the lowest. The minus sign flips the order, and `[:5]` keeps the top five.
- `table.argmax(axis=1)` gives, for each row, the column position of the highest value. That is each row's top neighbor.
- TF-IDF tables are **sparse**: they store only the values that are not zero. `.toarray()` turns a sparse result into an ordinary numpy table. Use it only on results that fit in memory.

The cell prints the shapes first. Then read the result row by row. Which pairs score higher than you expected?

In [ ]:
# Look up a sentence's text or note by its ID, for example sent_text["S01"].
sent_text = sentences.set_index("sentence_id")["text"]
sent_note = sentences.set_index("sentence_id")["note"]

# (11 x 384) @ (384 x 11) gives an 11 x 11 table: the cosine of every sentence with every other one.
cosines = sent_vectors @ sent_vectors.T
sent_sim = pd.DataFrame(cosines, index=sentences["sentence_id"], columns=sentences["sentence_id"])
print("sent_vectors:", sent_vectors.shape, "| sent_vectors.T:", sent_vectors.T.shape, "| cosine table:", sent_sim.shape)

# Each pair is (anchor sentence, a sentence written to go with it).
pairs = [("S01", "S02"), ("S01", "S03"), ("S01", "S04"), ("S01", "S05"),
         ("S06", "S07"), ("S06", "S08"), ("S09", "S10"), ("S09", "S11")]

# Build one table row per pair. sent_sim.loc[row, column] reads one cosine.
rows = []
for first_id, second_id in pairs:
    rows.append({
        "pair": first_id + "-" + second_id,
        "first": sent_text[first_id],
        "second": sent_text[second_id],
        "note": sent_note[second_id],
        "embedding_cosine": round(float(sent_sim.loc[first_id, second_id]), 2),
    })
pair_table = pd.DataFrame(rows)
pair_table

### 1.7 The same pairs with TF-IDF

Now we score the same pairs with the Module 4 method. We fit TF-IDF on these 11 sentences only, to show how word overlap behaves. When two sentences share no words, their TF-IDF cosine is 0.

In [ ]:
# Same TF-IDF setup as Module 4: fit_transform learns the words and returns one row per sentence.
sent_tfidf = TfidfVectorizer()
X_sent = sent_tfidf.fit_transform(sentences["text"])

# TF-IDF rows also have length 1, so X @ X.T gives cosines. .toarray() turns the sparse result into an ordinary table.
tfidf_sim = pd.DataFrame((X_sent @ X_sent.T).toarray(), index=sentences["sentence_id"], columns=sentences["sentence_id"])

# Read the TF-IDF cosine for each pair and add it as a new column.
tfidf_scores = []
for first_id, second_id in pairs:
    tfidf_scores.append(round(float(tfidf_sim.loc[first_id, second_id]), 2))
pair_table["tfidf_cosine"] = tfidf_scores
pair_table[["pair", "note", "tfidf_cosine", "embedding_cosine"]]

### 1.8 Load the customer messages and their vectors

These are the 24 customer messages from Lab 1. The saved vectors have 23 rows, not 24: FB022 has no text, so it has no vector. `dropna` keeps the 23 messages with text, in the same order as the vectors.

In [ ]:
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")

# Keep the messages that have text, and use the message ID as the row label.
with_text = feedback.dropna(subset=["text"]).set_index("feedback_id")

# One saved vector per message with text, in the same order.
fb_vectors = np.load(EMB / "feedback_minilm.npy").astype("float32")
print("shape:", fb_vectors.shape)

### 1.9 Nearest neighbors

A **nearest neighbor** is the message with the highest cosine to the one you start from. The helper below returns the top `k` neighbors with their scores and text, and leaves out the message itself.

In [ ]:
# A 23 x 23 cosine table, with the message IDs as row and column labels.
fb_sim = pd.DataFrame(fb_vectors @ fb_vectors.T, index=with_text.index, columns=with_text.index)

def fb_neighbors(feedback_id, sim, k=3):
    """Take a message ID and a cosine table; return the k closest other messages with cosine and text."""
    # The message's row of cosines, without the message itself.
    row = sim.loc[feedback_id].drop(feedback_id)
    # Sort from the highest cosine down and keep the first k.
    scores = row.sort_values(ascending=False).head(k)
    return pd.DataFrame({"cosine": scores.round(2), "text": with_text.loc[scores.index, "text"]})

print(with_text.loc["FB001", "text"])
fb_neighbors("FB001", fb_sim)

### 1.10 Compare with TF-IDF neighbors

Same message, same 23 messages, but TF-IDF features. Both methods rank FB018 ("missing charging cable") first. Compare the scores and the other neighbors. What does each method match on?

In [ ]:
# Fit TF-IDF on the 23 messages and build the same kind of cosine table.
fb_tfidf = TfidfVectorizer()
X_fb = fb_tfidf.fit_transform(with_text["text"])
fb_tfidf_sim = pd.DataFrame((X_fb @ X_fb.T).toarray(), index=with_text.index, columns=with_text.index)
# The same helper works with any cosine table.
fb_neighbors("FB001", fb_tfidf_sim)

### 1.11 A misleading neighbor

FB006 asks for help: the tablet will not start. Look at its closest neighbors. One of them says the problem is already **solved**. The embedding sees the same topic, but the service decision is the opposite.

In [ ]:
# Print the starting message, then its three closest messages by embedding cosine.
print(with_text.loc["FB006", "text"])
fb_neighbors("FB006", fb_sim)

### 1.12 Save your outputs

This writes two small CSV files to `outputs/m05`. The data files are not changed. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
pair_table.to_csv(OUTPUT_DIR / "sentence_pairs.csv", index=False)

# Collect the neighbors of FB001 and FB006 for both methods in one table.
neighbor_rows = []
for start in ["FB001", "FB006"]:
    for method, sim in [("embedding", fb_sim), ("tfidf", fb_tfidf_sim)]:
        # reset_index turns the neighbor IDs (the row labels) into a column.
        found = fb_neighbors(start, sim).reset_index(names="neighbor_id")
        # insert(0, ...) adds a column at the front.
        found.insert(0, "method", method)
        found.insert(0, "start_id", start)
        neighbor_rows.append(found)
pd.concat(neighbor_rows).to_csv(OUTPUT_DIR / "feedback_neighbors.csv", index=False)

print("saved:", OUTPUT_DIR / "sentence_pairs.csv")
print("saved:", OUTPUT_DIR / "feedback_neighbors.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. In 1.6, "The tablet will start." scores higher with S01 than "My tablet won't turn on." What does this mean for a queue of messages that need help?
2. Name one detail in the manifest that another analyst would need to reproduce or check these vectors.

**Your answer:**

## Part 2 Practice: real clothing reviews

Now use the saved vectors for the 22,634 clothing reviews you met in Lab 4. The same model made these vectors from the `review_text` column, without any cleaning. The merchandising team wants to find reviews that describe the same experience, even in different words. Read the data card `data/course/clothing_reviews_README.md` if you need a reminder of the columns.

### 2.1 Load the reviews and their vectors

The review vectors are in the same row order as `clothing_reviews.csv`, one vector per review. The IDs file lists the review ID of each vector row, in the same order. So the vector in row 5 belongs to the review in row 5. Never sort or filter one file without the other: a vector matched to the wrong review gives wrong neighbors and a wrong model card. The manifest describes how they were made. The IDs file also holds the number of tokens in each review, which we use in 2.2.

In [ ]:
reviews = pd.read_csv(DATA / "course/clothing_reviews.csv")

# The manifest, the IDs file (with token counts) and the vectors for the reviews.
review_manifest = json.loads((EMB / "clothing_minilm_manifest.json").read_text())
review_ids = pd.read_csv(EMB / "clothing_minilm_ids.csv")
review_vectors = np.load(EMB / "clothing_minilm.npy").astype("float32")
print("shape:", review_vectors.shape)

### 2.2 Were any reviews cut short?

The model reads at most 256 tokens. A token is a word or a piece of a word. Anything after that limit is ignored, so for a long text the vector describes only the start. The video showed that 42% of the CFPB complaints are longer than 256 tokens. Reviews are much shorter, so none should be cut.

In [ ]:
# Copy the token count of each review into the reviews table (same row order).
reviews["n_tokens"] = review_ids["n_tokens"]

# The manifest records how many reviews were longer than the limit.
print("reviews cut at 256 tokens:", review_manifest["rows_truncated"])
print("longest review in tokens:", reviews["n_tokens"].max())
reviews["n_tokens"].describe().round(1)

### 2.3 Search only among training reviews

We start from a `valid` review and look for neighbors among `train` reviews. This keeps the same split rules as Lab 4. The helper shows the cosine, department, star rating and the start of each neighbor's text.

The first line lets tables show up to 120 characters of text. Inside the helper, `train_vectors @ query` gives 13,581 cosines, one per training review. `np.argsort(-scores)[:k]` keeps the positions of the `k` highest.

In [ ]:
pd.set_option("display.max_colwidth", 120)  # show up to 120 characters of text in tables

# True for training reviews, False for the others, as a numpy array.
is_train = reviews["split"].eq("train").to_numpy()
# The training reviews, numbered 0, 1, 2, ... so that row i matches train_vectors[i].
train = reviews[is_train].reset_index(drop=True)
train_vectors = review_vectors[is_train]

# row_of[review_id] gives the row position of a review in `reviews` and `review_vectors`.
row_of = pd.Series(range(len(reviews)), index=reviews["review_id"])

def embedding_neighbors(review_id, k=5):
    """Take a review ID; return its k closest training reviews by embedding cosine."""
    query = review_vectors[row_of[review_id]]
    # One cosine per training review.
    scores = train_vectors @ query
    # Positions of the k highest scores, highest first.
    top = np.argsort(-scores)[:k]
    # .copy() makes a separate result table, so adding columns does not touch `train`.
    found = train.loc[top, ["review_id", "department", "rating"]].copy()
    # insert(1, ...) puts the cosine column in position 1, right after review_id.
    found.insert(1, "cosine", scores[top].round(2))
    # .str[:100] keeps the first 100 characters of each text.
    found["text_start"] = train.loc[top, "review_text"].str[:100]
    return found.reset_index(drop=True)

### 2.4 Neighbors for one review

Review 1466 praises the fit of a sweater but says the wool is too itchy. Read its neighbors. The closest one has the same complaint. Another says its wool is soft and *not* itchy: the same topic with the opposite verdict, as in 1.11.

In [ ]:
start_id = 1466
# Print the full text of the starting review, then its neighbors.
print(reviews.loc[row_of[start_id], "review_text"])
embedding_neighbors(start_id)

### 2.5 The same search with TF-IDF

We fit TF-IDF on the training reviews with the Lab 4 Practice settings (words and two-word phrases, `min_df=5`, no stop words), then use `transform` for the starting review. Compare the two lists of IDs.

In [ ]:
# Same settings as Lab 4 Practice.
review_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),  # words and two-word phrases
    min_df=5,            # keep terms that appear in at least 5 training reviews
    stop_words=None,     # do not remove stop words, including "not"
)
# fit_transform learns the vocabulary from the training reviews only.
X_train = review_tfidf.fit_transform(train["review_text"])

def tfidf_neighbors(review_id, k=5):
    """Take a review ID; return its k closest training reviews by TF-IDF cosine."""
    # transform uses the vocabulary learned above; it does not learn new words.
    query = review_tfidf.transform([reviews.loc[row_of[review_id], "review_text"]])
    # One cosine per training review. .ravel() turns the one-column table into a flat list.
    scores = (X_train @ query.T).toarray().ravel()
    top = np.argsort(-scores)[:k]
    found = train.loc[top, ["review_id", "department", "rating"]].copy()
    found.insert(1, "cosine", scores[top].round(2))
    found["text_start"] = train.loc[top, "review_text"].str[:100]
    return found.reset_index(drop=True)

tfidf_neighbors(start_id)

### 2.6 Dense vectors from your own word counts

Dense vectors do not always come from a pretrained model. **Truncated SVD** squeezes the TF-IDF columns into 100 combined dimensions. These "latent" vectors are learned only from our training reviews, so they know nothing about language outside this data. A pretrained embedding model learned from a very large amount of other text before it ever saw our reviews.

The last line prints the share of the spread (variance) in the TF-IDF table that the 100 dimensions keep. A small share means the SVD keeps the strongest patterns of word use and drops most of the detail.

In [ ]:
from sklearn.decomposition import TruncatedSVD

# 100 combined dimensions; random_state fixes the seed, so everyone gets the same result.
svd = TruncatedSVD(n_components=100, random_state=42)
Z_train = svd.fit_transform(X_train)
print("TF-IDF shape:", X_train.shape)
print("SVD shape:", Z_train.shape)
# explained_variance_ratio_ holds one share per dimension; their sum is the share kept.
print("share of TF-IDF variance kept:", round(svd.explained_variance_ratio_.sum(), 3))

### 2.7 A cosine score is not a probability

Here we take 20,000 random pairs of reviews and compare scores for pairs from the same department with pairs from different departments. The video showed the same check on CFPB complaints. Again the ranges overlap a lot. A score of 0.5 does not mean "50% likely to be about the same thing". It only ranks pairs for this model and this data.

Department agreement is a rough check of topic matching. It does not tell us whether two reviews describe the same problem. Read the texts to judge that.

In [ ]:
# Draw 20,000 random pairs of row positions. Seed 0 gives everyone the same pairs.
rng = np.random.default_rng(0)
first = rng.integers(0, len(reviews), 20000)
second = rng.integers(0, len(reviews), 20000)

# Drop the few pairs where a review was paired with itself.
keep = first != second
first = first[keep]
second = second[keep]

# Multiply two vectors number by number and add up: that is the dot product, so the cosine.
pair_cos = (review_vectors[first] * review_vectors[second]).sum(axis=1)

# True where both reviews of a pair come from the same department.
department = reviews["department"].to_numpy()
same_department = department[first] == department[second]

# np.quantile gives the 10th percentile, the median and the 90th percentile.
# ~ means "not", so pair_cos[~same_department] keeps the pairs from different departments.
percentiles = [0.1, 0.5, 0.9]
summary = pd.DataFrame({
    "same department": np.quantile(pair_cos[same_department], percentiles),
    "different department": np.quantile(pair_cos[~same_department], percentiles),
}, index=["10th percentile", "median", "90th percentile"])
summary.round(2)

### Exercise 1: compare two neighbor lists

Review 13907 says a white shirt is see-through. Show its top 5 neighbors with `embedding_neighbors` and with `tfidf_neighbors`. Then count how many review IDs appear in **both** lists.

Hint: `set(list_a) & set(list_b)` keeps the items two lists share.

In [ ]:
# Exercise 1: your code here
# TODO: get the top 5 neighbors with embedding_neighbors(13907) and print the table.
# TODO: get the top 5 neighbors with tfidf_neighbors(13907) and print the table.
# TODO: put the "review_id" column of each table in a set, keep the shared IDs with &,
#       and print how many there are.

### Exercise 2: a close neighbor with a very different rating

Review 15751 gave a dress 2 stars: it runs small and is shorter than expected. Show its top 10 embedding neighbors. Find one neighbor rated **5 stars**, and print the full text of both reviews. What do the two reviews share, and where do they differ?

In [ ]:
# Exercise 2: your code here
# TODO: get the top 10 embedding neighbors of review 15751 (use k=10) and print them.
# TODO: keep the neighbors with rating 5 and pick one of their review IDs.
# TODO: print the full review_text of review 15751 and of that 5-star neighbor
#       (use reviews.loc[row_of[review_id], "review_text"]).

### Exercise 3: how often does the top neighbor share the department?

For the first 1,000 `valid` reviews, find each review's single closest `train` review, once with embeddings and once with TF-IDF. Report the share of these validation reviews whose closest neighbor is from the **same department**, for each method, rounded to three decimals.

Hints:
- `valid_rows = np.flatnonzero(reviews["split"].eq("valid").to_numpy())[:1000]` gives the row positions of the first 1,000 validation reviews.
- `review_vectors[valid_rows] @ train_vectors.T` gives all their cosines with the training reviews: a 1,000 × 13,581 table with one row per validation review. `.argmax(axis=1)` finds each row's top neighbor.
- For TF-IDF, use `review_tfidf.transform(...)` on `reviews["review_text"].iloc[valid_rows]`, then `(X_valid @ X_train.T).toarray()`.
- We use 1,000 reviews, not all 4,528, so the cosine tables fit in a laptop's memory.
- Compare departments as numpy arrays, so they are matched by position and not by row label. For embeddings:

```python
# Use arrays so departments are compared by position.
valid_departments = reviews["department"].to_numpy()[valid_rows]
neighbor_departments = train["department"].to_numpy()[top_emb]
# True counts as 1, so the mean is the matching share.
share = (neighbor_departments == valid_departments).mean()
```

Then repeat the same steps for TF-IDF. Remember that the same department is only a rough sign of the same topic, not of the same problem.

In [ ]:
# Exercise 3: your code here
# TODO: get the row positions of the first 1,000 validation reviews (see the first hint).
# TODO: embeddings: compute their cosines with train_vectors and take .argmax(axis=1).
# TODO: TF-IDF: transform their review_text, compute cosines with X_train, take .argmax(axis=1).
# TODO: for each method, compare the neighbor's department with the review's own department,
#       and print the share that match, rounded to three decimals.

### Exercise 4: a model card for the review vectors

A **model card** is a short note about a model: which model made the numbers, what went in, and where the model is weak. Another analyst reads it before reusing your vectors. This exercise takes about 10 minutes.

Run the cell below. It prints the manifest fields for the review vectors you used in Part 2. Then fill in the model card in the next cell, one line per item (4–6 lines in total):

1. **Model:** the model name and its revision.
2. **Input:** which file and column went in, and whether the text was cleaned.
3. **Cut text:** the token limit, and how many reviews were cut. Would that change for longer texts, such as complaints?
4. **Output:** the number of rows and the numbers per vector.
5. **One limitation for review text:** name one, and support it with a result from this lab (for example 1.11, 2.4 or Exercise 2).

In [ ]:
# Print the manifest fields you need for the model card.
for key in ["model", "model_revision", "input", "max_tokens", "rows_truncated", "rows", "dimensions"]:
    print(f"{key}: {review_manifest[key]}")
print("longest review in tokens (from the IDs file):", review_ids["n_tokens"].max())

**Your model card:**

- Model:
- Input:
- Cut text:
- Output:
- One limitation for review text:

### Practice interpretation

Write 3–5 sentences:

- In Exercise 1, how much did the two neighbor lists overlap? What does each method seem to match on?
- In Exercise 2, the two reviews are close but their ratings differ a lot. What does the embedding seem to match on: the topic or the opinion? Quote the text that supports your view.
- Use Exercise 3 and 2.7: would you trust a cosine of 0.80 to decide that two reviews describe the same problem? What would you check first?

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Three representations side by side

Repeat Exercise 3 for the SVD vectors from 2.6, so you can compare word counts, latent vectors and a pretrained embedding on the same task. SVD vectors are not length 1, so we scale them first.

In [ ]:
# np.flatnonzero gives the positions where the condition is True; keep the first 1,000.
valid_rows = np.flatnonzero(reviews["split"].eq("valid").to_numpy())[:1000]
valid_departments = reviews["department"].to_numpy()[valid_rows]

# TF-IDF first, then the SVD learned in 2.6 (transform only, no new fitting).
X_valid_tfidf = review_tfidf.transform(reviews["review_text"].iloc[valid_rows])
Z_valid = svd.transform(X_valid_tfidf)

# Divide each row by its length, so every vector has length 1 and @ gives cosines.
# keepdims=True keeps the lengths as a column, so each row is divided by its own length.
Z_train_unit = Z_train / np.linalg.norm(Z_train, axis=1, keepdims=True)
Z_valid_unit = Z_valid / np.linalg.norm(Z_valid, axis=1, keepdims=True)

# For each validation review, the position of its closest training review.
top_svd = (Z_valid_unit @ Z_train_unit.T).argmax(axis=1)

# True where the neighbor has the same department; the mean of True/False values is the share of True.
same_svd = train["department"].to_numpy()[top_svd] == valid_departments
print("SVD: share with the same department:", round(same_svd.mean(), 3))

### B. Topic or opinion?

Exercise 2 suggests that embeddings match what a review is about more than how the customer felt. Test this on the same 1,000 validation reviews. Put each rating in a band: low (1–2 stars), middle (3) or high (4–5). How often does the top embedding neighbor fall in the same band? Compare it with a lazy rule that always says "high".

In [ ]:
def band(stars):
    """Take an array of star ratings; return "low" (1-2), "middle" (3) or "high" (4-5) for each."""
    # np.select checks the conditions in order and uses "high" when none is true.
    return np.select([stars <= 2, stars == 3], ["low", "middle"], "high")

# The closest training review for each of the 1,000 validation reviews (embeddings).
top_emb = (review_vectors[valid_rows] @ train_vectors.T).argmax(axis=1)

# The rating band of each validation review and of its top neighbor.
valid_band = band(reviews["rating"].to_numpy()[valid_rows])
neighbor_band = band(train["rating"].to_numpy()[top_emb])
print("embeddings: same rating band:", round((neighbor_band == valid_band).mean(), 3))
print("always 'high':", round((valid_band == "high").mean(), 3))

### C. Encode your own sentences

This part needs the `sentence-transformers` package, which is not in the course environment. It downloads the model (about 90 MB) the first time. If you want to try it, run `pip install sentence-transformers` in the Anaconda Prompt or Terminal with the `ban6403` environment active. Then set `RUN_ENCODER_CHALLENGE = True` in the cell below. The cell compares your own vector for S01 with the saved one.

The switch stays `False` by default, so Run All never downloads a model.

In [ ]:
# Change to True only when you want to run Challenge C (it downloads the model).
RUN_ENCODER_CHALLENGE = False

if not RUN_ENCODER_CHALLENGE:
    print("Challenge C is switched off. This part is optional.")
else:
    # The package is optional, so the cell prints a note instead of failing when it is missing.
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("sentence-transformers is not installed. This part is optional.")
    else:
        model = SentenceTransformer(manifest["model"], revision=manifest["model_revision"])
        # normalize_embeddings=True scales each vector to length 1, like the saved vectors.
        # encode returns one row per sentence; [0] takes the first (and only) row.
        mine = model.encode(["The tablet will not start."], normalize_embeddings=True)[0]
        print("cosine with the saved S01 vector:", round(float(mine @ sent_vectors[0]), 4))
        print(model.encode(["Your own sentence here."], normalize_embeddings=True).shape)

## Further reading

- [Sentence Transformers: semantic textual similarity](https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html)
- [Model card: all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [scikit-learn: TruncatedSVD (latent semantic analysis)](https://scikit-learn.org/stable/modules/decomposition.html#truncated-singular-value-decomposition-and-latent-semantic-analysis)

Next, Module 6 uses embeddings like these to search for evidence and measures how good the results are.